<a href="https://colab.research.google.com/github/Santiago-Parada/Public-Projects/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import fitz
import re
import pandas as pd

def text_process(file,tmp):
  doc = fitz.open(file)

  text = ""

  for page in doc:
      text += page.get_text("text") + "\n"

  doc.close()

  text = text.replace("\r", "")
  text = re.sub(r"\n{3,}", "\n\n", text)

  inicio_matches = list(
      re.finditer(
          r"I\.1\.\s+Entorno económico y composición del aseguramiento",
          text
      )
  )

  if len(inicio_matches) >= 2:
      text = text[inicio_matches[1].start():]

  patron = re.compile(
      r"(?m)^([IVXLCDM]+\.\d+\.\s+.+)$"
  )

  matches = list(patron.finditer(text))

  secciones = []

  for i, match in enumerate(matches):

      inicio = match.start()

      fin = (
          matches[i + 1].start()
          if i < len(matches) - 1
          else len(text)
      )

      bloque = text[inicio:fin].strip()
      bloque = re.sub(r"(\w)-\s+(\w)", r"\1\2", bloque)
      bloque = re.sub(r"\s+", " ", bloque).strip()
      encabezado = match.group(1).strip()

      codigo = re.match(
          r"([IVXLCDM]+\.\d+)\.",
          encabezado
      ).group(1)

      secciones.append({
          "seccion": encabezado,
          "documento": bloque
      })

  df = pd.DataFrame(secciones)
  df["periodo"]=tmp
  return df[["periodo","seccion","documento"]]

df1=text_process("19-informe-completo.pdf","2024-2025")
df2=text_process("19-informe-completo 23-24.pdf","2023-2024")
df3=text_process("19-informe-completo 22-23.pdf","2022-2023")
df4=text_process("19-informe-completo 21-22.pdf","2021-2022")
df5=text_process("19-informe-completo 20-21.pdf","2020-2021")
df6=text_process("19-informe-completo 19-20.pdf","2019-2020")


df = pd.concat([df1, df2,df3,df4,df5,df6], ignore_index=True)

# Guardar
df.to_parquet(
    "imss_data.parquet",
    index=False
)

print(df.head())
print("Total secciones:", len(df))

     periodo                                            seccion  \
0  2024-2025  I.1. Entorno económico y composición del asegu...   
1  2024-2025                         I.2. Cobertura poblacional   
2  2024-2025  I.3. El reto del envejecimiento poblacional y ...   
3  2024-2025  II.1. Resultados con base en los Estados Finan...   
4  2024-2025              II.2. Información presupuestaria 2024   

                                           documento  
0  I.1. Entorno económico y composición del asegu...  
1  I.2. Cobertura poblacional Esta sección descri...  
2  I.3. El reto del envejecimiento poblacional y ...  
3  II.1. Resultados con base en los Estados Finan...  
4  II.2. Información presupuestaria 2024 Durante ...  
Total secciones: 526


In [ ]:

pd.set_option("display.max_colwidth", None)
df[df["periodo"]=="2020-2021"]["seccion"]

,seccion
301,I.1. Estados financieros
302,I.2. Proyecciones financieras a corto plazo
303,I.3. Proyecciones financieras de largo plazo
304,II.1. Ingresos del IMSS
305,II.2. Aseguramiento en el IMSS
...,...
425,XIII.3. ESTADO DE LAS FINANZAS INSTITUCIONALES Y PROPUESTAS
426,C.1. DESCRIPCIÓN DE LOS BENEFICIOS VALUADOS
427,C.2. INFORMACIÓN AL CIERRE DE 2020
428,D.1. ELEMENTOS DEL PROCESO DE INVERSIÓN


SEGUNDA ENTREGA

In [2]:
# !python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 32.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import pandas as pd
from collections import Counter
from collections import defaultdict
from spacy import displacy,load
import numpy as np
doc_es=pd.read_parquet("sample_data/imss_data.parquet")
nlp_es = load("es_core_news_sm")

In [20]:
doc_es3 = doc_es[["periodo"]].copy()
doc_es3["doc"] = doc_es["documento"].apply(nlp_es)

In [21]:

tokens_periodo = defaultdict(list)

for _, fila in doc_es3.iterrows():
    periodo = fila["periodo"]
    doc = fila["doc"]

    tokens = [
        token.text.lower()
        for token in doc
        if not token.is_space
    ]

    tokens_periodo[periodo].extend(tokens)

resultados = []

for periodo, tokens in tokens_periodo.items():
    total = len(tokens)
    unicos = len(set(tokens))
    diversidad = unicos / total

    resultados.append({
        "periodo": periodo,
        "tokens_totales": total,
        "tokens_unicos": unicos,
        "diversidad_lexica": diversidad
    })

diversidad_df = pd.DataFrame(resultados).sort_values("periodo", ascending=False)

diversidad_df

,periodo,tokens_totales,tokens_unicos,diversidad_lexica
0,2024-2025,156367,14051,0.089859
1,2023-2024,171437,14196,0.082806
2,2022-2023,165505,13680,0.082656
3,2021-2022,154173,16518,0.107139
4,2020-2021,242440,18418,0.075969
5,2019-2020,208107,15801,0.075927


In [22]:
top20 = []

for periodo, tokens in tokens_periodo.items():
    frecuencia = Counter(tokens)

    for token, frec in frecuencia.most_common(20):
        top20.append({
            "periodo": periodo,
            "token": token,
            "frecuencia": frec
        })

top20_df = pd.DataFrame(top20)

top20_df

,periodo,token,frecuencia
0,2024-2025,de,12677
1,2024-2025,",",6329
2,2024-2025,la,4541
3,2024-2025,y,4295
4,2024-2025,.,4230
...,...,...,...
115,2019-2020,para,1853
116,2019-2020,con,1762
117,2019-2020,al,1711
118,2019-2020,(,1634


In [23]:
top20_df.to_excel("tokens_frecuencia.xlsx",index=False)

In [24]:
resultado = []

for periodo, tokens in tokens_periodo.items():

    token_mas_largo = max(tokens, key=len)

    resultado.append({
        "periodo": periodo,
        "token_mas_largo": token_mas_largo,
        "longitud": len(token_mas_largo)
    })

token_largo_df = pd.DataFrame(resultado).sort_values("periodo", ascending=False)

token_largo_df

,periodo,token_mas_largo,longitud
0,2024-2025,https://www.banxico.org.mx/publicaciones-y-pre...,146
1,2023-2024,www.gob.mx/conapo/documentos/bases-de-datos-de...,147
2,2022-2023,https://datos.gob.mx/busca/dataset/proyeccione...,114
3,2021-2022,nəƺɴȵƺƴǣƭǣȓȇƴƺǽȓɀƭƺȸɏǣˡƭəƴȓɀƴƺǣȇƭəȵəƭǣƴə...,114
4,2020-2021,https://datos.gob.mx/busca/dataset/proyeccione...,113
5,2019-2020,http://www.imss.gob.mx/conoce-alimss/memoria-e...,61


In [25]:
entidades_periodo = {}

for _, fila in doc_es3.iterrows():

    periodo = fila["periodo"]
    doc = fila["doc"]

    if periodo not in entidades_periodo:
        entidades_periodo[periodo] = {}

    for ent in doc.ents:

        etiqueta = ent.label_

        if etiqueta not in entidades_periodo[periodo]:
            entidades_periodo[periodo][etiqueta] = 0

        entidades_periodo[periodo][etiqueta] += 1

In [26]:
entidades_df = (
    pd.DataFrame(entidades_periodo)
      .fillna(0)
      .astype(int)
      .T
)

entidades_df

,MISC,ORG,PER,LOC
2024-2025,4382,2069,1685,2781
2023-2024,5014,2623,1657,3575
2022-2023,5121,2453,1397,3298
2021-2022,4753,1915,1274,2747
2020-2021,8238,3517,1908,4696
2019-2020,6043,2386,2112,4587


In [27]:
entidades_df.to_excel("tokens_p_tipo_entidad.xlsx")

In [28]:
for periodo in ["2024-2025","2023-2024","2022-2023"]:

  subset = doc_es3[doc_es3["periodo"] == periodo]

  for doc in subset["doc"].head(3):
      displacy.render(doc, style="ent", jupyter=True)

In [29]:
from spacy import displacy

my_text = "IMSS"
max_muestras = 5
contador = 0

for idx, row in doc_es3.iterrows():
    doc = row["doc"]

    for token in doc:

        if token.text.lower() == my_text.lower():

            sent = token.sent
            displacy.render(sent, style="ent", jupyter=True)

            contador += 1

            if contador >= max_muestras:
                break

    if contador >= max_muestras:
        break

In [30]:


pos_cols = [
    "ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM",
    "PART","PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"
]

pos_por_periodo = {
    periodo: {pos: 0 for pos in pos_cols}
    for periodo in doc_es3["periodo"].unique()
}

In [31]:
for _, row in doc_es3.iterrows():

    periodo = row["periodo"]
    doc = row["doc"]

    for token in doc:
        pos = token.pos_

        if pos in pos_por_periodo[periodo]:
            pos_por_periodo[periodo][pos] += 1

In [32]:
doc_es3_pos = pd.DataFrame(pos_por_periodo).T

doc_es3_pos

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
2024-2025,11794,27857,1673,1151,5205,15211,10,31394,14039,29,4359,18807,16412,1132,514,6780,0
2023-2024,11852,31422,1946,1327,6025,16684,14,33239,15600,86,4980,22264,16867,1287,523,7321,0
2022-2023,11567,30317,1854,1327,5644,16479,10,32299,15375,76,4980,20692,15943,1271,506,7165,0
2021-2022,10394,26600,1732,1291,4860,15163,31,28863,15256,54,4704,20900,15078,1180,716,7349,2
2020-2021,16731,44554,2728,2330,8229,24509,5,46690,22393,63,5968,32090,22598,1819,761,10972,0
2019-2020,14027,38146,2385,1713,7123,20460,5,39682,18717,68,6092,27800,20556,1472,604,9256,1


In [33]:
doc_es3_pos.to_excel("tokens_p_tipo_pos.xlsx")

In [34]:
top5_pos = (
    doc_es3_pos
    .apply(lambda row: row.sort_values(ascending=False).head(5), axis=1)
)

In [35]:
top5_pos_df = top5_pos.stack().reset_index()

top5_pos_df.columns = ["periodo", "POS", "frecuencia"]

top5_pos_df

,periodo,POS,frecuencia
0,2024-2025,ADP,27857.0
1,2024-2025,DET,15211.0
2,2024-2025,NOUN,31394.0
3,2024-2025,PROPN,18807.0
4,2024-2025,PUNCT,16412.0
5,2023-2024,ADP,31422.0
6,2023-2024,DET,16684.0
7,2023-2024,NOUN,33239.0
8,2023-2024,PROPN,22264.0
9,2023-2024,PUNCT,16867.0


In [36]:
top5_pivot = top5_pos_df.pivot(
    index="periodo",
    columns="POS",
    values="frecuencia"
)

top5_pivot

POS,ADP,DET,NOUN,NUM,PROPN,PUNCT
periodo,,,,,,
2019-2020,38146.0,20460.0,39682.0,NaN,27800.0,20556.0
2020-2021,44554.0,24509.0,46690.0,NaN,32090.0,22598.0
2021-2022,26600.0,15163.0,28863.0,15256.0,20900.0,NaN
2022-2023,30317.0,16479.0,32299.0,NaN,20692.0,15943.0
2023-2024,31422.0,16684.0,33239.0,NaN,22264.0,16867.0
2024-2025,27857.0,15211.0,31394.0,NaN,18807.0,16412.0
